# Evaluación HellaSwag

Este notebook carga el checkpoint del proyecto y selecciona, entre cuatro terminaciones, la que tiene mayor probabilidad según el modelo. Solo requiere `torch`, `tiktoken` y `requests`.

In [1]:
from pathlib import Path
import json
import sys

import requests
import tiktoken
import torch
import torch.nn.functional as F

ROOT = Path.cwd().resolve()
if not (ROOT / 'Foundation Model').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'Foundation Model'))
from Transformer_arquitectures import GPTModel
#model_name = 'model.pth'
model_name = "gpt2-fineweb-124m-step-120000.pt"

CHECKPOINT = ROOT / 'Model Checkpoints' / model_name
assert CHECKPOINT.is_file(), f'No se encontró el checkpoint: {CHECKPOINT}'
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')

default_config = {'vocab_size': 50257, 'context_length': 256, 'emb_dim': 768, 'n_heads': 12, 'n_layers': 12, 'drop_rate': 0.0, 'qkv_bias': False}
checkpoint = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint else checkpoint
config = {**default_config, **(checkpoint.get('config', {}) if isinstance(checkpoint, dict) else {})}
model = GPTModel(config).to(device).eval()
model.load_state_dict(state_dict)
tokenizer = tiktoken.get_encoding('gpt2')
print(f'Modelo cargado en {device}; contexto={config["context_length"]}')


Modelo cargado en mps; contexto=512


In [2]:
def score_completion(prompt_ids, completion_ids):
    tokens = (prompt_ids + completion_ids)[-config['context_length']:]
    completion_start = max(1, len(tokens) - len(completion_ids))
    input_ids = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)
    with torch.inference_mode():
        logits = model(input_ids[:, :-1])
    log_probs = F.log_softmax(logits, dim=-1).gather(-1, input_ids[:, 1:].unsqueeze(-1)).squeeze(-1)[0]
    completion_log_probs = log_probs[completion_start - 1:]
    return completion_log_probs.sum().item(), completion_log_probs.mean().item()

def load_hellaswag():
    url = 'https://raw.githubusercontent.com/rowanz/hellaswag/master/data/hellaswag_val.jsonl'
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    return [json.loads(line) for line in response.text.splitlines() if line]

def evaluate(max_examples=100):
    examples = load_hellaswag()[:max_examples] if max_examples is not None else load_hellaswag()
    correct_raw = correct_norm = 0
    for number, example in enumerate(examples, start=1):
        prompt = tokenizer.encode(f"{example['ctx_a']} {example['ctx_b']}")
        scores = [score_completion(prompt, tokenizer.encode(' ' + ending)) for ending in example['endings']]
        label = int(example['label'])
        correct_raw += max(range(4), key=lambda i: scores[i][0]) == label
        correct_norm += max(range(4), key=lambda i: scores[i][1]) == label
        if number % 100 == 0 or number == len(examples):
            print(f'{number}/{len(examples)}  acc={correct_raw / number:.4f}  acc_norm={correct_norm / number:.4f}')
    return {'model_name' : model_name, "Benchmark" : "HellaSwag" , 'examples': len(examples), 'acc': correct_raw / len(examples), 'acc_norm': correct_norm / len(examples)}

# Usa None para los 10 042 ejemplos de validación.
results = evaluate(max_examples=100)
results


100/100  acc=0.3700  acc_norm=0.3700


{'model_name': 'gpt2-fineweb-124m-step-120000.pt',
 'Benchmark': 'HellaSwag',
 'examples': 100,
 'acc': 0.37,
 'acc_norm': 0.37}

In [3]:
from pathlib import Path
import json

output_dir = ROOT / "benchmarks_results" / model_name 

output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "benchmarks_result_Hellaswag.txt"
output_file.write_text(
    json.dumps(results, ensure_ascii=False, indent=4),
    encoding="utf-8"
)

146